# MLP PyTorch — Telco Customer Churn

Construção, treinamento e avaliação de uma rede neural **Multilayer Perceptron (MLP)** com PyTorch para previsão de churn.

**Abordagem:** Comparamos 2 versões da MLP, ambas com mini-batches (DataLoader):
- **MLP-v1**: arquitetura simples, early stopping na loss de treino
- **MLP-v2**: arquitetura mais profunda, split treino/validação/teste e early stopping na **loss de validação** — previne overfitting

## 0. Imports e carregamento dos dados

In [56]:
import random
import hashlib
import logging
import pickle
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, recall_score,
                             classification_report)
from torch.utils.data import TensorDataset, DataLoader

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

DATA_PATH = "../data/telco_churn_clean.csv"
df = pd.read_csv(DATA_PATH)

# Versão do dataset via hash MD5 — rastreabilidade no MLflow
with open(DATA_PATH, "rb") as f:
    dataset_hash = hashlib.md5(f.read()).hexdigest()

drop_cols = ['CustomerID', 'Country', 'State', 'City', 'Lat Long',
             'Churn Reason', 'Churn Score', 'Churn Value', 'CLTV']

X = df.drop(columns=drop_cols + ['Churn Label'])
y = (df['Churn Label'] == 'Yes').astype(int)
X = pd.get_dummies(X)

logger.info("Dataset carregado: %d linhas | %d features | dataset_hash=%s",
            df.shape[0], X.shape[1], dataset_hash)

INFO - Dataset carregado: 7043 linhas | 50 features | dataset_hash=8bdf2e12867b9a60298ad9c28c5ea9bf


## 1. Reprodutibilidade — Seed global

Fixamos a seed em todos os geradores de aleatoriedade relevantes:
- `random` e `numpy` — para operações Python/NumPy
- `torch.manual_seed` — para operações PyTorch na CPU
- `cudnn.deterministic` — garante determinismo nas operações de GPU

In [57]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

logger.info("Seed fixado: %d", SEED)

INFO - Seed fixado: 42


## 2. Configuração do MLflow

Usamos backend SQLite em vez do FileStore (deprecated desde fev/2026).
Todos os experimentos deste notebook são registrados no mesmo experimento `telco-churn-baselines`,
permitindo comparação direta entre baselines e versões da MLP.

In [58]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("telco-churn-baselines")
logger.info("MLflow configurado — experimento: telco-churn-baselines")

INFO - MLflow configurado — experimento: telco-churn-baselines


## 3. Divisão dos dados

Usamos divisão em **3 conjuntos**:
- **Teste (20%)** — hold-out final, usado apenas para avaliação dos dois modelos
- **Treino (64%)** + **Validação (16%)** — usados durante o treinamento do MLP-v2

`stratify=y` garante que a proporção de churn (~26,5%) seja mantida em todos os splits.

> Esta divisão 3 vias é o padrão recomendado para redes neurais: o conjunto de
> validação monitora o overfitting **durante** o treino sem contaminar a avaliação final.

In [59]:
# Split 1: separa hold-out de teste (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Split 2: separa validação do treino (20% do restante → 16% do total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

logger.info("Treino: %d | Validação: %d | Teste: %d",
            len(X_train), len(X_val), len(X_test))

INFO - Treino: 4507 | Validação: 1127 | Teste: 1409


## 4. Normalização

Redes neurais são sensíveis à escala dos dados — features com valores muito maiores
dominam os gradientes e dificultam o aprendizado.

O `StandardScaler` transforma cada feature para média 0 e desvio padrão 1.

> **Importante:** o scaler é ajustado **apenas no treino** (`fit_transform`) e aplicado
> no val e teste (`transform`). Isso evita vazamento de informação.

In [60]:
# Normalizar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

## 5. Conversão para tensores PyTorch

PyTorch trabalha apenas com seu próprio formato — o **tensor**, uma matriz otimizada
para rodar em GPU. Convertemos os arrays NumPy para `FloatTensor` (float32),
que é o tipo padrão esperado pela maioria das operações PyTorch.

In [61]:
X_train_t = torch.FloatTensor(X_train)
X_val_t   = torch.FloatTensor(X_val)
X_test_t  = torch.FloatTensor(X_test)
y_train_t = torch.FloatTensor(y_train.values)
y_val_t   = torch.FloatTensor(y_val.values)
y_test_t  = torch.FloatTensor(y_test.values)
#Divisao: 64% Treino | 16% Validação | 20% Teste
logger.info("Shape treino: %s | val: %s | teste: %s",
            X_train_t.shape, X_val_t.shape, X_test_t.shape)

INFO - Shape treino: torch.Size([4507, 50]) | val: torch.Size([1127, 50]) | teste: torch.Size([1409, 50])


## 6. Função de perda compartilhada

**`BCEWithLogitsLoss`** — combina Sigmoid + Binary Cross-Entropy em uma única operação numericamente estável.

**`pos_weight`** — como o dataset é desbalanceado (~26,5% de churn), penalizamos o modelo mais quando ele erra um churn real (falso negativo) do que quando erra um não-churn (falso positivo).
O peso é calculado como `n_negativos / n_positivos ≈ 2,77`.

In [62]:
# Por termos um desbalanceamento nos dados, com 5174 ''não cancela'' e 1869''cancela'' estou adicionando weight para a rede aprender que errar um churn é mais grave do que errar um não churn
pos_weight = torch.tensor([len(y_train[y_train==0]) / len(y_train[y_train==1])])
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

logger.info("Peso da classe positiva: %.2f", pos_weight.item())

INFO - Peso da classe positiva: 2.77


## 7. MLP-v1 — Arquitetura Simples com Mini-Batches

Construímos a MLP como uma classe Python herdando de `nn.Module` — padrão PyTorch.

**Arquitetura v1:** `input → 64 → 32 → 1`
- Treino com **DataLoader (batch_size=32)** para comparação justa com o v2
- Early stopping baseado na **loss de treino**
- **ReLU** — função de ativação não-linear que evita o problema do gradiente que some
- **Dropout (30%)** — desliga aleatoriamente 30% dos neurônios para regularização
- **Sigmoid** — converte a saída para probabilidade entre 0 e 1

In [63]:
class ChurnMLP(nn.Module): #Criando rede neural como classe
    def __init__(self, input_dim #Numero de features
    ):
        super().__init__()
        self.network = nn.Sequential( #Define as camadas em que os dados passarão em sequencia
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1), #retorna 1 unico valor
            nn.Sigmoid() #aqui vira probabilidade
        )

    def forward(self, x):
        return self.network(x)


dataset_v1    = TensorDataset(X_train_t, y_train_t)
dataloader_v1 = DataLoader(dataset_v1, batch_size=32, shuffle=True)

model     = ChurnMLP(input_dim=X_train_t.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

logger.info("Arquitetura MLP-v1:\n%s", model) #print da estrutura

epochs   = 300
patience = 10  # para se não melhorar em 10 épocas seguidas
best_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for X_batch, y_batch in dataloader_v1:
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader_v1)

    # Early stopping baseado na loss de treino
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), "../models/mlp_v1.pt")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        logger.info("Early stopping na época %d", epoch + 1)
        break

    if (epoch + 1) % 10 == 0:
        logger.info("Época %d | Loss treino: %.4f", epoch + 1, avg_loss)

INFO - Arquitetura MLP-v1:
ChurnMLP(
  (network): Sequential(
    (0): Linear(in_features=50, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
    (7): Sigmoid()
  )
)
INFO - Época 10 | Loss treino: 0.8904
INFO - Época 20 | Loss treino: 0.8847
INFO - Época 30 | Loss treino: 0.8799
INFO - Época 40 | Loss treino: 0.8750
INFO - Época 50 | Loss treino: 0.8707
INFO - Época 60 | Loss treino: 0.8706
INFO - Early stopping na época 61


## 8. Avaliação — MLP-v1

Avaliamos o modelo no conjunto de teste e registramos as métricas no MLflow.

> **Nota:** A avaliação acontece **antes** de abrir o `mlflow.start_run`, para garantir
> que as métricas logadas são os valores reais calculados — e não valores fixos no código.

In [64]:
model.load_state_dict(torch.load("../models/mlp_v1.pt", weights_only=True))
model.eval()
with torch.no_grad():
    preds_v1      = model(X_test_t).squeeze()
    preds_binary  = (preds_v1 >= 0.5).float()

auc_v1       = roc_auc_score(y_test, preds_v1.numpy())
pr_auc_v1    = average_precision_score(y_test, preds_v1.numpy())
f1_v1        = f1_score(y_test, preds_binary.numpy())
precision_v1 = precision_score(y_test, preds_binary.numpy(), zero_division=0)
recall_v1    = recall_score(y_test, preds_binary.numpy())

with mlflow.start_run(run_name="MLP-v1"):
    mlflow.log_param("hidden_layers",    [64, 32])
    mlflow.log_param("dropout",          0.3)
    mlflow.log_param("lr",               0.001)
    mlflow.log_param("patience",         10)
    mlflow.log_param("batch_size",       32)
    mlflow.log_param("early_stop_on",    "train_loss")
    mlflow.log_param("seed",             SEED)
    mlflow.log_param("dataset_version",  dataset_hash)
    mlflow.log_metric("auc_roc",         auc_v1)
    mlflow.log_metric("pr_auc",          pr_auc_v1)
    mlflow.log_metric("f1",              f1_v1)
    mlflow.log_metric("precision",       precision_v1)
    mlflow.log_metric("recall",          recall_v1)

logger.info("MLP-v1 — AUC: %.3f | PR-AUC: %.3f | F1: %.3f | Precision: %.3f | Recall: %.3f",
            auc_v1, pr_auc_v1, f1_v1, precision_v1, recall_v1)
print(classification_report(y_test, preds_binary.numpy(), target_names=['Ficou', 'Cancelou']))

INFO - MLP-v1 — AUC: 0.835 | PR-AUC: 0.615 | F1: 0.609 | Precision: 0.571 | Recall: 0.652


              precision    recall  f1-score   support

       Ficou       0.87      0.82      0.84      1035
    Cancelou       0.57      0.65      0.61       374

    accuracy                           0.78      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.78      0.78      1409



## 9. MLP-v2 — Arquitetura Mais Profunda com Early Stopping na Validação

O diferencial do MLP-v2 é o **early stopping baseado na loss de validação** — não na de treino.
Isso é fundamental: parar quando o modelo para de generalizar, não quando para de memorizar.

| Parâmetro | v1 | v2 |
|-----------|-----|----|
| Arquitetura | 64 → 32 | 128 → 64 → 32 |
| Dropout | 30% | 20% |
| Learning rate | 0.001 | 0.001 |
| Patience | 10 | 20 |
| Batch size | 32 | 32 |
| Early stopping | Loss de **treino** | Loss de **validação** |

> O early stopping na validação é o padrão profissional para redes neurais —
> garante que salvamos o modelo no ponto de melhor generalização.

In [65]:
class ChurnMLPv2(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


dataset_v2    = TensorDataset(X_train_t, y_train_t)
dataloader_v2 = DataLoader(dataset_v2, batch_size=32, shuffle=True)

model_v2  = ChurnMLPv2(input_dim=X_train_t.shape[1])
optimizer = torch.optim.Adam(model_v2.parameters(), lr=0.001)

logger.info("Arquitetura MLP-v2:\n%s", model_v2)

epochs   = 300
patience = 20  # para se não melhorar em 20 épocas seguidas
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    # --- Treino ---
    model_v2.train()
    epoch_loss = 0
    for X_batch, y_batch in dataloader_v2:
        optimizer.zero_grad()
        outputs = model_v2(X_batch).squeeze()
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # --- Validação ---
    model_v2.eval()
    with torch.no_grad():
        val_outputs = model_v2(X_val_t).squeeze()
        val_loss    = criterion(val_outputs, y_val_t).item()

    avg_train_loss = epoch_loss / len(dataloader_v2)

    # Early stopping baseado na loss de VALIDAÇÃO — previne overfitting
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model_v2.state_dict(), "../models/mlp_v2.pt")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        logger.info("Early stopping na época %d | val_loss: %.4f", epoch + 1, val_loss)
        break

    if (epoch + 1) % 10 == 0:
        logger.info("Época %d | Loss treino: %.4f | Loss val: %.4f",
                    epoch + 1, avg_train_loss, val_loss)

INFO - Arquitetura MLP-v2:
ChurnMLPv2(
  (network): Sequential(
    (0): Linear(in_features=50, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)
INFO - Época 10 | Loss treino: 0.6405 | Loss val: 0.6741
INFO - Época 20 | Loss treino: 0.5763 | Loss val: 0.6984
INFO - Época 30 | Loss treino: 0.5332 | Loss val: 0.8161
INFO - Early stopping na época 32 | val_loss: 0.8011


## 10. Avaliação — MLP-v2

Recarregamos o **melhor checkpoint** (época com menor val_loss) para a avaliação final.

In [66]:
model_v2.load_state_dict(torch.load("../models/mlp_v2.pt", weights_only=True))
model_v2.eval()
with torch.no_grad():
    preds_v2     = torch.sigmoid(model_v2(X_test_t)).squeeze()
    preds_bin_v2 = (preds_v2 >= 0.5).float()

auc_v2       = roc_auc_score(y_test, preds_v2.numpy())
pr_auc_v2    = average_precision_score(y_test, preds_v2.numpy())
f1_v2        = f1_score(y_test, preds_bin_v2.numpy())
precision_v2 = precision_score(y_test, preds_bin_v2.numpy(), zero_division=0)
recall_v2    = recall_score(y_test, preds_bin_v2.numpy())

with mlflow.start_run(run_name="MLP-v2"):
    mlflow.log_param("hidden_layers",   [128, 64, 32])
    mlflow.log_param("dropout",         0.2)
    mlflow.log_param("lr",              0.001)
    mlflow.log_param("patience",        20)
    mlflow.log_param("batch_size",      32)
    mlflow.log_param("early_stop_on",   "val_loss")
    mlflow.log_param("seed",            SEED)
    mlflow.log_param("dataset_version", dataset_hash)
    mlflow.log_metric("auc_roc",   auc_v2)
    mlflow.log_metric("pr_auc",    pr_auc_v2)
    mlflow.log_metric("f1",        f1_v2)
    mlflow.log_metric("precision", precision_v2)
    mlflow.log_metric("recall",    recall_v2)

logger.info("MLP-v2 — AUC: %.3f | PR-AUC: %.3f | F1: %.3f | Precision: %.3f | Recall: %.3f",
            auc_v2, pr_auc_v2, f1_v2, precision_v2, recall_v2)
print(classification_report(y_test, preds_bin_v2.numpy(), target_names=['Ficou', 'Cancelou']))

INFO - MLP-v2 — AUC: 0.844 | PR-AUC: 0.629 | F1: 0.625 | Precision: 0.503 | Recall: 0.824


              precision    recall  f1-score   support

       Ficou       0.92      0.71      0.80      1035
    Cancelou       0.50      0.82      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.81      0.74      0.75      1409



## 11. Comparação Final e Seleção do Modelo

Comparativo completo no conjunto de **hold-out (teste 20%)** — 5 métricas técnicas:

| Modelo | AUC-ROC | PR-AUC | F1 | Precision | Recall | Early Stop |
|--------|---------|--------|-----|-----------|--------|------------|
| Dummy (CV) | 0.500 | — | 0.000 | — | — | — |
| Logistic Regression (CV) | 0.858 | — | 0.619 | — | — | — |
| Decision Tree (CV) | 0.845 | — | 0.582 | — | — | — |
| Random Forest (CV) | 0.859 | 0.678 | 0.598 | 0.670 | 0.540 | — |
| MLP-v1 | 0.835 | 0.615 | 0.609 | 0.571 | 0.652 | Loss treino |
| **MLP-v2** | **0.844** | **0.629** | **0.625** | **0.503** | **0.824** | **Loss validação** |

### Por que o MLP-v2 usa early stopping na validação?

Com mini-batches, a **loss de treino cai muito mais rápido** e pode atingir valores
muito baixos (~0.30) sem que o modelo generalize bem — foi exatamente o que aconteceu
quando monitorávamos a loss de treino. O MLP-v2 exibiu **overfitting severo**:
rodou as 300 épocas, loss de treino caiu a 0.30, mas todas as métricas no teste pioraram.

Ao monitorar a **loss de validação**, o modelo para de treinar no ponto ótimo de
generalização — resultando em melhores métricas no conjunto de teste.

In [67]:
# Tabela comparativa dinâmica com os valores reais calculados
print("=" * 85)
print(f"{'Modelo':<30} {'AUC':>7} {'PR-AUC':>7} {'F1':>7} {'Precision':>10} {'Recall':>8}")
print("=" * 85)
print(f"{'Dummy (CV)':<30} {'0.500':>7} {'  —':>7} {'0.000':>7} {'   —':>10} {'  —':>8}")
print(f"{'Logistic Regression (CV)':<30} {'0.858':>7} {'  —':>7} {'0.619':>7} {'   —':>10} {'  —':>8}")
print(f"{'Decision Tree (CV)':<30} {'0.845':>7} {'  —':>7} {'0.582':>7} {'   —':>10} {'  —':>8}")
print(f"{'Random Forest (CV)':<30} {'0.859':>7} {'0.678':>7} {'0.598':>7} {'0.670':>10} {'0.540':>8}")
print(f"{'MLP-v1 (train loss ES)':<30} {auc_v1:>7.3f} {pr_auc_v1:>7.3f} {f1_v1:>7.3f} {precision_v1:>10.3f} {recall_v1:>8.3f}")
print(f"{'MLP-v2 (val loss ES) ✅':<30} {auc_v2:>7.3f} {pr_auc_v2:>7.3f} {f1_v2:>7.3f} {precision_v2:>10.3f} {recall_v2:>8.3f}")
print("=" * 85)

# Seleção automática com base no AUC (critério principal)
if auc_v2 >= auc_v1:
    melhor_modelo  = model_v2
    melhor_nome    = "MLP-v2"
    melhor_path    = "../models/mlp_v2.pt"
    logger.info("Modelo selecionado: MLP-v2 (AUC %.3f >= MLP-v1 AUC %.3f)", auc_v2, auc_v1)
else:
    melhor_modelo  = model
    melhor_nome    = "MLP-v1"
    melhor_path    = "../models/mlp_v1.pt"
    logger.info("Modelo selecionado: MLP-v1 (AUC %.3f >= MLP-v2 AUC %.3f)", auc_v1, auc_v2)

INFO - Modelo selecionado: MLP-v2 (AUC 0.844 >= MLP-v1 AUC 0.835)


Modelo                             AUC  PR-AUC      F1  Precision   Recall
Dummy (CV)                       0.500       —   0.000          —        —
Logistic Regression (CV)         0.858       —   0.619          —        —
Decision Tree (CV)               0.845       —   0.582          —        —
Random Forest (CV)                0.859   0.678   0.598      0.670    0.540
MLP-v1 (train loss ES)           0.835   0.615   0.609      0.571    0.652
MLP-v2 (val loss ES) ✅           0.844   0.629   0.625      0.503    0.824


## 12. Análise de Custo — Trade-off FP vs FN

Em churn, os dois tipos de erro têm **custos de negócio muito diferentes**:

| Tipo de Erro | O que significa | Custo estimado |
|--------------|-----------------|----------------|
| **Falso Positivo (FP)** | Modelo prevê churn, mas cliente não ia cancelar → oferta de retenção desnecessária | R$ 50 (desconto/campanha) |
| **Falso Negativo (FN)** | Modelo não identifica um churner → cliente é perdido | R$ 200 (≈ 3 meses × mensalidade média) |

> **Custo total = FP × R$ 50 + FN × R$ 200**

O FN é **4x mais caro** que o FP — justificando a estratégia de priorizar Recall
(capturar mais churners) mesmo com queda na Precision.

**Métrica de negócio: Custo de Churn Evitado** — diferença entre o custo sem nenhum modelo
e o custo com o modelo preditivo.

In [68]:
from sklearn.metrics import confusion_matrix

# Custo por erro de negócio
CUSTO_FP = 50   # R$ — oferta de retenção para quem não ia cancelar
CUSTO_FN = 200  # R$ — perda de um cliente (≈ 3 meses × mensalidade média)

# Confusion matrix: [[TN, FP], [FN, TP]]
cm_v1 = confusion_matrix(y_test, preds_binary.numpy())
cm_v2 = confusion_matrix(y_test, preds_bin_v2.numpy())

tn_v1, fp_v1, fn_v1, tp_v1 = cm_v1.ravel()
tn_v2, fp_v2, fn_v2, tp_v2 = cm_v2.ravel()

# Custo total sem modelo (abordar nenhum cliente)
fn_sem_modelo = y_test.sum()  # todos os churners são perdidos
custo_sem_modelo = fn_sem_modelo * CUSTO_FN

custo_v1 = fp_v1 * CUSTO_FP + fn_v1 * CUSTO_FN
custo_v2 = fp_v2 * CUSTO_FP + fn_v2 * CUSTO_FN

print("=" * 60)
print(f"{'Cenário':<30} {'FP':>5} {'FN':>5} {'Custo Total':>12}")
print("=" * 60)
print(f"{'Sem modelo':<30} {0:>5} {fn_sem_modelo:>5} R$ {custo_sem_modelo:>9,.0f}")
print(f"{'MLP-v1':<30} {fp_v1:>5} {fn_v1:>5} R$ {custo_v1:>9,.0f}")
print(f"{'MLP-v2 ✅':<30} {fp_v2:>5} {fn_v2:>5} R$ {custo_v2:>9,.0f}")
print("=" * 60)

economia_v1 = custo_sem_modelo - custo_v1
economia_v2 = custo_sem_modelo - custo_v2

print(f"\nEconomia com MLP-v1 vs sem modelo: R$ {economia_v1:,.0f}")
print(f"Economia com MLP-v2 vs sem modelo: R$ {economia_v2:,.0f}")
print(f"Vantagem do MLP-v2 sobre MLP-v1:   R$ {economia_v2 - economia_v1:,.0f}")

logger.info("Custo MLP-v1: R$ %d | Custo MLP-v2: R$ %d | Economia v2: R$ %d",
            custo_v1, custo_v2, economia_v2 - economia_v1)

INFO - Custo MLP-v1: R$ 35150 | Custo MLP-v2: R$ 28400 | Economia v2: R$ 6750


Cenário                           FP    FN  Custo Total
Sem modelo                         0   374 R$    74,800
MLP-v1                           183   130 R$    35,150
MLP-v2 ✅                         304    66 R$    28,400

Economia com MLP-v1 vs sem modelo: R$ 39,650
Economia com MLP-v2 vs sem modelo: R$ 46,400
Vantagem do MLP-v2 sobre MLP-v1:   R$ 6,750


### Interpretação da análise de custo

O MLP-v2 reduz o custo operacional de duas formas:
- **Mais churners capturados (Recall 82,4% vs 65,2%)** → menos FN, evitando perdas de receita
- **Custo de FP administrável** → ofertas de retenção para não-churners têm custo fixo e limitado

O alto Recall do MLP-v2 é **especialmente valioso em telecomunicações**, onde o custo de
aquisição de um novo cliente é tipicamente 5x–7x o custo de retenção de um cliente existente.
Mesmo abordando mais clientes sem necessidade (FP), a economia gerada pelo menor número
de clientes perdidos (FN) supera o custo das campanhas de retenção.

## 13. Salvando artefatos do modelo final

Salvamos o **melhor modelo** (`mlp_best.pt`), o **scaler** e as **colunas de treino** —
os três são necessários para a API de inferência reconstruir exatamente o mesmo
pré-processamento aplicado durante o treino.

O modelo final é selecionado automaticamente com base no maior AUC no conjunto de teste.

In [69]:
# Recarrega o melhor checkpoint e salva como mlp_best.pt
melhor_modelo.load_state_dict(torch.load(melhor_path, weights_only=True))
torch.save(melhor_modelo.state_dict(), "../models/mlp_best.pt")
logger.info("Modelo final: %s → salvo em ../models/mlp_best.pt", melhor_nome)

# Salvar o scaler
with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Salvar as colunas usadas no treino
with open("../models/feature_columns.json", "w") as f:
    json.dump(list(X.columns), f)

logger.info("Scaler e colunas salvos em ../models/")

INFO - Modelo final: MLP-v2 → salvo em ../models/mlp_best.pt
INFO - Scaler e colunas salvos em ../models/
